### Gradio

In [1]:
pip install gradio


Note: you may need to restart the kernel to use updated packages.


In [2]:
import gradio as gr
def greet(name, intensity):
    return "Hello, ", name, "!" *int(intensity)
demo = gr.Interface(
    fn = greet,
    inputs=["text", "slider"],
    outputs = ["text"],
)
demo.launch(server_name="127.0.0.1", server_port = 7860)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [3]:
%pip install transformers torch




Note: you may need to restart the kernel to use updated packages.


In [4]:
import gradio as gr
from transformers import BlipProcessor, BlipForConditionalGeneration

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

def generate_caption(image):
    # Now directly using the PIL Image object
    inputs = processor(images=image, return_tensors="pt")
    outputs = model.generate(**inputs)
    caption = processor.decode(outputs[0], skip_special_tokens=True)
    return caption

def caption_image(image):
    """
    Takes a PIL Image input and returns a caption.
    """
    try:
        caption = generate_caption(image)
        return caption
    except Exception as e:
        return f"An error occurred: {str(e)}"

iface = gr.Interface(
    fn=caption_image,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="Image Captioning with BLIP",
    description="Upload an image to generate a caption."
)

iface.launch(server_name="127.0.0.1")


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


### Image Classification In PyTorch

#### 1. Setting Up Image Classification Model

In [5]:
import torch
from torchvision.models import resnet18, ResNet18_Weights
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1).eval()

#### 2: Defining a predict function

In [6]:
import torch
import requests
from torchvision import transforms
import gradio as gr

## Load Model

# Download human-readable labels for ImageNet
response = requests.get("https://git.io/JJkYN")
labels = [l.strip() for l in response.text.split("\n") if l.strip()]

# Define image preprocessing (IMPORTANT for ResNet)
transform = transforms.Compose([ transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )
])

## Processing Function

def predict(inp):

    # A. Prepare input
    # preprocess image
    inp = transform(inp).unsqueeze(0)

    # A. Prepare input
    # ensure model runs in inference mode
    with torch.no_grad():
        prediction = torch.nn.functional.softmax(model(inp)[0], dim=0)

    # C. Decode result
    # map predictions to labels
    confidences = {
        labels[i]: float(prediction[i])
        for i in range(len(labels))
    }

    return confidences


## UI Connect

# Gradio Interface

iface1=gr.Interface(
    fn=predict,
    inputs=gr.Image(type="pil"),
    outputs=gr.Label(num_top_classes=3),
    examples=["Image.png","Tiger.png"]).launch()



* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
